In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"  # avoids an openmp dll clash between anaconda mkl and pip torch on windows

import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.ndimage import gaussian_filter
from skimage.transform import resize
from skimage.metrics import peak_signal_noise_ratio, structural_similarity


In [2]:
# frame-wise spatial baseline only - no temporal context, no respiratory-phase
# conditioning, no segmentation inputs, no neighbouring-frame or pca features
VAL_SUBJECT = "TV7_20220315_sorted_UCL"
TEST_SUBJECT = "TV8_20220307_sorted_UCL"  # internal synthetic test set, not the real iqt lr-hr evaluation

CROP_SIZE = 80
SCALE_FACTOR = 1.5  # hr/lr spacing ratio baked into the export - fixed, has physical meaning

SIGMA_REF = 1.0  # sigma used to build the export's fixed lr_synth_frames
SIGMA_MIN = 0.7
SIGMA_MAX = 1.3

CHANNELS = [16, 32, 64, 128]  # unet encoder channel counts, one per downsampling level

EPOCHS = 40
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
SEED = 0

OUTPUT_DIR = Path("Outputs")
EXPORT_PATH = OUTPUT_DIR / "tv_hr_crop_synthetic_lr_export.npz"
CHECKPOINT_PATH = OUTPUT_DIR / "spatial_baseline_best.pt"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True, warn_only=True)


In [3]:
export_data = np.load(EXPORT_PATH, allow_pickle=True)
print("export arrays:", list(export_data.files))

hr_frames = export_data["hr_frames"]
included = export_data["included"]
subject = export_data["subject"]
sl = export_data["slice"]
dyn = export_data["dynamic"]
phase_bin = export_data["phase_bin"]

print("total exported frames:", len(hr_frames))
print("frame shape:", hr_frames.shape[1:])

# lr_synth_frames is a fixed-sigma reference artefact from the export, not the training
# input - training lr is regenerated per-frame below with a sampled sigma instead

hr_frames = hr_frames[included]
subject = subject[included]
sl = sl[included]
dyn = dyn[included]
phase_bin = phase_bin[included]

print("included frame count after filtering:", len(hr_frames))


export arrays: ['hr_frames', 'lr_synth_frames', 'subject', 'slice', 'dynamic', 'phase_bin', 'phase_direction', 'anchor_row', 'anchor_col', 'crop_origin_row', 'crop_origin_col', 'included']


total exported frames: 9120
frame shape: (80, 80)
included frame count after filtering: 7030


In [4]:
# no "echo" field exists in the export at all - by construction (data_prep.ipynb step 8)
# every exported frame was pulled with echo_num=1, so te1 is the only echo present
assert "echo" not in export_data.files
echoes_used = ["TE1"]
print("echoes present in export metadata: none recorded (te1-only by construction)")
print("echo(es) used for this baseline:", echoes_used)


echoes present in export metadata: none recorded (te1-only by construction)
echo(es) used for this baseline: ['TE1']


In [5]:
all_subjects = sorted(set(subject.tolist()))
train_subjects = [s for s in all_subjects if s not in (VAL_SUBJECT, TEST_SUBJECT)]

# split by subject, never by frame - adjacent dynamics are highly correlated, a random
# frame split would leak information between train/val/test. 8 subjects total makes this
# a small-sample split
train_mask = np.isin(subject, train_subjects)
val_mask = subject == VAL_SUBJECT
test_mask = subject == TEST_SUBJECT

train_idx = np.where(train_mask)[0]
val_idx = np.where(val_mask)[0]
test_idx = np.where(test_mask)[0]

print("train subjects:", train_subjects)
print("val subject:", VAL_SUBJECT)
print("internal synthetic test subject:", TEST_SUBJECT)
print("train/val/test frame counts:", len(train_idx), len(val_idx), len(test_idx))


train subjects: ['TV1_20220222_sorted_UCL', 'TV2_20220222_sorted_UCL', 'TV3_20220208_sorted_UCL', 'TV4_20220204_sorted_UCL', 'TV5_20220307_sorted_UCL', 'TV6_20220307_sorted_UCL']
val subject: TV7_20220315_sorted_UCL
internal synthetic test subject: TV8_20220307_sorted_UCL
train/val/test frame counts: 5320 570 1140


In [6]:
def compute_percentile_scale(frames, subject, sl, low=1, high=99):
    # one (lo, hi) pair per subject/slice time series, not per frame - keeps genuine
    # temporal intensity changes across dynamics instead of erasing them
    scale = {}
    for key in set(zip(subject.tolist(), sl.tolist())):
        series = frames[(subject == key[0]) & (sl == key[1])]
        scale[key] = tuple(np.percentile(series, [low, high]))
    return scale


def normalise(img, lo, hi):
    return np.clip((img - lo) / max(hi - lo, 1e-6), 0.0, 1.0)


def denormalise(img, lo, hi):
    # maps a normalised prediction back to the original hr intensity scale
    return img * (hi - lo) + lo


# scaling statistics come from the hr series only, then get reused on both hr and its
# synthetic lr counterpart so the two share one transformation
percentile_scale = compute_percentile_scale(hr_frames, subject, sl)
print(f"percentile scaling computed for {len(percentile_scale)} subject/slice series")


percentile scaling computed for 37 subject/slice series


In [7]:
# copied directly from data_prep.ipynb step 8 (not imported) so the forward model is
# fully visible here
def degrade_hr_to_lr(image, blur_sigma=SIGMA_REF, downsample_factor=SCALE_FACTOR):
    blurred = gaussian_filter(image.astype(np.float64), sigma=blur_sigma)
    size = image.shape[0]
    small_size = max(1, int(round(size / downsample_factor)))
    downsampled = resize(blurred, (small_size, small_size), order=3, mode="edge",
                          anti_aliasing=False, preserve_range=True)
    upsampled = resize(downsampled, (size, size), order=3, mode="edge",
                        anti_aliasing=False, preserve_range=True)
    return upsampled


In [8]:
class SpatialPairDataset(torch.utils.data.Dataset):
    def __init__(self, indices, sigma_min, sigma_max, fixed_sigma=None):
        self.indices = indices
        self.sigma_min = sigma_min
        self.sigma_max = sigma_max
        self.fixed_sigma = fixed_sigma  # val/test pass this so metrics are comparable across epochs

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        idx = self.indices[i]
        hr = hr_frames[idx]
        lo, hi = percentile_scale[(subject[idx], sl[idx])]

        # training samples a fresh sigma every load, val/test always use sigma_ref
        sigma = self.fixed_sigma if self.fixed_sigma is not None else np.random.uniform(self.sigma_min, self.sigma_max)
        lr = degrade_hr_to_lr(hr, blur_sigma=sigma)

        hr_t = torch.from_numpy(normalise(hr, lo, hi)).float().unsqueeze(0)
        lr_t = torch.from_numpy(normalise(lr, lo, hi)).float().unsqueeze(0)
        return lr_t, hr_t


In [9]:
train_ds = SpatialPairDataset(train_idx, SIGMA_MIN, SIGMA_MAX)
val_ds = SpatialPairDataset(val_idx, SIGMA_MIN, SIGMA_MAX, fixed_sigma=SIGMA_REF)
test_ds = SpatialPairDataset(test_idx, SIGMA_MIN, SIGMA_MAX, fixed_sigma=SIGMA_REF)  # internal synthetic test set - not used below

train_loader = torch.utils.data.DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = torch.utils.data.DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = torch.utils.data.DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)


In [10]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class UNet(nn.Module):
    # 80x80 input, four downsampling levels (80 is divisible by 16): 80/40/20/10/5
    def __init__(self, channels):
        super().__init__()
        c1, c2, c3, c4 = channels
        self.pool = nn.MaxPool2d(2)

        self.enc1 = ConvBlock(1, c1)
        self.enc2 = ConvBlock(c1, c2)
        self.enc3 = ConvBlock(c2, c3)
        self.enc4 = ConvBlock(c3, c4)
        self.bottleneck = ConvBlock(c4, c4)

        self.up4 = nn.ConvTranspose2d(c4, c4, 2, stride=2)
        self.dec4 = ConvBlock(c4 + c4, c3)
        self.up3 = nn.ConvTranspose2d(c3, c3, 2, stride=2)
        self.dec3 = ConvBlock(c3 + c3, c2)
        self.up2 = nn.ConvTranspose2d(c2, c2, 2, stride=2)
        self.dec2 = ConvBlock(c2 + c2, c1)
        self.up1 = nn.ConvTranspose2d(c1, c1, 2, stride=2)
        self.dec1 = ConvBlock(c1 + c1, c1)

        self.out_conv = nn.Conv2d(c1, 1, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        b = self.bottleneck(self.pool(e4))

        d4 = self.dec4(torch.cat([self.up4(b), e4], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.out_conv(d1)


# residual prediction: model(lr) predicts hr - lr, not hr directly
model = UNet(CHANNELS)


In [11]:
def psnr_batch(pred, target):
    pred_np, target_np = pred.detach().cpu().numpy(), target.detach().cpu().numpy()
    return float(np.mean([peak_signal_noise_ratio(target_np[i, 0], pred_np[i, 0], data_range=1.0) for i in range(pred_np.shape[0])]))


def ssim_batch(pred, target):
    pred_np, target_np = pred.detach().cpu().numpy(), target.detach().cpu().numpy()
    return float(np.mean([structural_similarity(target_np[i, 0], pred_np[i, 0], data_range=1.0) for i in range(pred_np.shape[0])]))


In [12]:
def run_train_epoch(model, loader, optimizer):
    model.train()
    total_loss, n = 0.0, 0
    for lr, hr in loader:
        pred_residual = model(lr)
        loss = F.l1_loss(pred_residual, hr - lr)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * lr.size(0)
        n += lr.size(0)
    return total_loss / n


def evaluate(model, loader):
    # also reports the degraded-input (lr vs hr) baseline so it's clear whether the
    # unet actually improves on the synthetic lr input, not just its absolute score
    model.eval()
    total = {"loss": 0.0, "psnr": 0.0, "ssim": 0.0, "baseline_psnr": 0.0, "baseline_ssim": 0.0}
    n = 0
    with torch.no_grad():
        for lr, hr in loader:
            pred_residual = model(lr)
            loss = F.l1_loss(pred_residual, hr - lr)
            prediction = (lr + pred_residual).clamp(0.0, 1.0)

            bs = lr.size(0)
            total["loss"] += loss.item() * bs
            total["psnr"] += psnr_batch(prediction, hr) * bs
            total["ssim"] += ssim_batch(prediction, hr) * bs
            total["baseline_psnr"] += psnr_batch(lr, hr) * bs
            total["baseline_ssim"] += ssim_batch(lr, hr) * bs
            n += bs
    return {k: v / n for k, v in total.items()}


def train_model(model, train_loader, val_loader, epochs, lr, checkpoint_path):
    # internal test subject is never passed in here - checkpoint selection only ever
    # sees train/val
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    best_val_loss = float("inf")
    history = []

    for epoch in range(epochs):
        train_loss = run_train_epoch(model, train_loader, optimizer)
        val_metrics = evaluate(model, val_loader)

        print(f"epoch {epoch + 1}/{epochs}  train_l1={train_loss:.4f}  val_l1={val_metrics['loss']:.4f}  "
              f"val_psnr={val_metrics['psnr']:.2f}  val_ssim={val_metrics['ssim']:.4f}  "
              f"baseline_psnr={val_metrics['baseline_psnr']:.2f}  baseline_ssim={val_metrics['baseline_ssim']:.4f}")

        if val_metrics["loss"] < best_val_loss:
            best_val_loss = val_metrics["loss"]
            torch.save(model.state_dict(), checkpoint_path)

        history.append({"epoch": epoch + 1, "train_loss": train_loss, **val_metrics})
    return history


In [13]:
# cpu-only smoke test - fixed 50-frame subset, fixed sigma_ref degradation, 2 epochs.
# checks the training code works, not whether real training would converge - real
# training is not started in this notebook
torch.manual_seed(SEED)

smoke_idx = train_idx[:50]
smoke_ds = SpatialPairDataset(smoke_idx, SIGMA_MIN, SIGMA_MAX, fixed_sigma=SIGMA_REF)
smoke_loader = torch.utils.data.DataLoader(smoke_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

smoke_model = UNet(CHANNELS).to("cpu")
smoke_optimizer = torch.optim.Adam(smoke_model.parameters(), lr=LEARNING_RATE)

smoke_losses = []
last_pred_shape, last_hr_shape = None, None
for epoch in range(2):
    epoch_loss, n = 0.0, 0
    for lr_batch, hr_batch in smoke_loader:
        pred_residual = smoke_model(lr_batch)
        loss = F.l1_loss(pred_residual, hr_batch - lr_batch)

        smoke_optimizer.zero_grad()
        loss.backward()
        smoke_optimizer.step()

        epoch_loss += loss.item() * lr_batch.size(0)
        n += lr_batch.size(0)
        last_pred_shape, last_hr_shape = pred_residual.shape, hr_batch.shape
    smoke_losses.append(epoch_loss / n)

assert all(np.isfinite(smoke_losses)), "smoke test produced a non-finite loss"
assert smoke_losses[-1] < smoke_losses[0], "smoke test loss did not decrease across epochs"
assert last_pred_shape == last_hr_shape, "model output shape does not match hr target shape"
smoke_passed = True


In [14]:
n_params = sum(p.numel() for p in UNet(CHANNELS).parameters() if p.requires_grad)

print("smoke-test epoch losses:", smoke_losses)
print("smoke test passed:", smoke_passed)
print("total trainable parameters:", n_params)
print("train subjects:", train_subjects)
print("val subject:", VAL_SUBJECT)
print("internal synthetic test subject:", TEST_SUBJECT)
print("train frames:", len(train_idx))
print("val frames:", len(val_idx))
print("internal test frames:", len(test_idx))
print("total included frame count:", len(hr_frames))
print("echo(es) used:", echoes_used)
print("model input shape: (batch, 1, 80, 80)")
print("SIGMA_REF:", SIGMA_REF, " training sigma range:", (SIGMA_MIN, SIGMA_MAX))


smoke-test epoch losses: [0.13721487879753114, 0.09667954683303832]
smoke test passed: True
total trainable parameters: 924785
train subjects: ['TV1_20220222_sorted_UCL', 'TV2_20220222_sorted_UCL', 'TV3_20220208_sorted_UCL', 'TV4_20220204_sorted_UCL', 'TV5_20220307_sorted_UCL', 'TV6_20220307_sorted_UCL']
val subject: TV7_20220315_sorted_UCL
internal synthetic test subject: TV8_20220307_sorted_UCL
train frames: 5320
val frames: 570
internal test frames: 1140
total included frame count: 7030
echo(es) used: ['TE1']
model input shape: (batch, 1, 80, 80)
SIGMA_REF: 1.0  training sigma range: (0.7, 1.3)
